# Synthetic photometry

This offline tutorial applies packaged filter responses to a smooth illustrative spectrum, builds an `SED`, and propagates a magnitude uncertainty. The continuum is teaching data rather than a calibrated stellar model.

In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np

from speclib import Filter, SED, Spectrum, apply_filter, mag_to_flux

## A spectrum that covers the filters

The wavelength range covers Bessell V and the 2MASS J/H bands. `apply_filter` does not validate full band coverage, so checking this is the caller's responsibility.

In [ ]:
wavelength = np.linspace(3000, 25_000, 11_001) * u.AA
flux_unit = u.erg / (u.s * u.cm**2 * u.AA)
continuum = 2e-9 * (wavelength.to_value(u.AA) / 5500) ** -2
broad_feature = 1 - 0.2 * np.exp(-0.5 * ((wavelength.value - 12_000) / 900) ** 2)
spectrum = Spectrum(spectral_axis=wavelength, flux=continuum * broad_feature * flux_unit)

## Apply one filter

A `Filter` stores effective wavelength, bandwidth, zeropoint data, and a dimensionless response curve. When shapes differ, `apply_filter` resamples that response onto the spectrum and mutates `filter.response`.

In [ ]:
j_filter = Filter("2MASS J")
j_flux = apply_filter(spectrum, j_filter)
print("effective wavelength:", j_filter.wl_eff)
print("bandwidth:", j_filter.bandwidth)
print("mean filtered flux density:", j_flux)

## Build an SED

`SED` repeats the same calculation for a list of filters and records the tabulated effective wavelengths and bandwidths alongside the resulting flux densities.

In [ ]:
filters = [Filter(name) for name in ("Bessell V", "2MASS J", "2MASS H")]
sed = SED(spectrum, filters)
for filt, wave, flux in zip(filters, sed.wavelength, sed.flux):
    print(f"{filt.name:10s} {wave:10.1f}  {flux:.4g}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.loglog(spectrum.wavelength, spectrum.flux, color="0.65", label="illustrative spectrum")
ax.scatter(sed.wavelength, sed.flux, color="C3", zorder=3, label="synthetic SED")
ax.set(xlabel="Wavelength [Angstrom]", ylabel=r"Flux density [erg s$^{-1}$ cm$^{-2}$ Angstrom$^{-1}$]")
ax.legend();

## Convert a magnitude and propagate its uncertainty

`mag_to_flux` samples both magnitude and the filter's tabulated zeropoint uncertainty, returning a mean and standard deviation. It currently uses NumPy's process-wide random generator, so this example sets a seed immediately before the call.

In [ ]:
np.random.seed(72)
mean_flux, flux_uncertainty = mag_to_flux(10.0, Filter("2MASS J"), mag_err=0.03, nsamples=20_000)
print(f"flux from magnitude: {mean_flux:.4g} +/- {flux_uncertainty:.2g}")

`apply_filter` and `SED` do not propagate spectral uncertainties and do not implement detector-specific photon-counting conventions. `SEDGrid` currently supports only PHOENIX. Validate response normalization, band coverage, and uncertainty assumptions for precision applications.